In [ ]:
import sys
if 'google.colab' in sys.modules:
    !rm -rf /content/fuse-qa
    !git clone https://github.com/MinaGabriel/fuse-qa.git
    !export HF_HUB_ENABLE_HF_TRANSFER=1
    %cd /content/fuse-qa
    !pip install -q -e .

Cloning into 'fuse-qa'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 129 (delta 49), reused 113 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 656.29 KiB | 20.51 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/fuse-qa
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fuseqa (pyproject.toml) ... done


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
from huggingface_hub import login

from fuseqa import *

# ─────────────────────────────────────────────────────────────
# Setup
# ─────────────────────────────────────────────────────────────

logging.set_verbosity_error()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# MODEL_NAME = "meta-llama/Meta-Llama-3-8B"
# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-7B"
# MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
# MODEL_NAME = "google/gemma-2-2b-it"
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
MODEL_NAME = "openai/gpt-oss-20b"
# MODEL_NAME ="Qwen/Qwen2.5-32B-Instruct"


print("GPUs:", torch.cuda.device_count())

# ─────────────────────────────────────────────────────────────
# HuggingFace Auth (optional)
# ─────────────────────────────────────────────────────────────

if (token := os.getenv("HF_TOKEN")):
    login(token=token)
    print("HF login: done")
else:
    print("HF login: skipped")

# ─────────────────────────────────────────────────────────────
# Load Model + Tokenizer
# ─────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": 0},          # force all weights to cuda:0
    #torch_dtype=torch.float16,   # or torch.bfloat16 if supported
)

model.eval()

# ─────────────────────────────────────────────────────────────
# Device Info
# ─────────────────────────────────────────────────────────────

device = next(model.parameters()).device
print("Model device:", device)

if hasattr(model, "hf_device_map"):
    print("Device map:", model.hf_device_map)



GPUs: 1
HF login: skipped


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model device: cuda:0


In [ ]:
from datasets import load_dataset
ds = load_dataset("MinaGabriel/popqa-retrieval20-with-sre")["test"].select(range(250))

In [ ]:
def prompt_fn(question: str, context: str, use_context: bool):
    if use_context:
        return (
            f"Context: {context}\n\n"
            f"Q: {question}\n"
            "A (one word from context):"
        )
    else:
        return (
            f"Q: {question}\n"
            "A (one word):"
        )

In [ ]:
from tqdm.auto import tqdm  # better for notebooks
import os
import time
from contextlib import nullcontext

# -------------------------
# Config
# -------------------------
TOP_K = 3
SRE_RETR_COL   = "retrieved_docs_sre"
BASE_RETR_COL  = "retrieved_docs"
SRE_SCORE_TH   = 0.90
RUN_TYPE = "FUSEQA-SRE"
USE_CONTEXT = RUN_TYPE in ("FUSEQA", "FUSEQA-SRE")
RESULTS_DIR = "results"
WRITE_OUTPUTS = True  # set True to write JSONL
PRINT_PROMPTS = False  # set True to print prompts and predictions during generation

os.makedirs(RESULTS_DIR, exist_ok=True)

file_name = hf_model_to_filename(MODEL_NAME + "-" + RUN_TYPE)
outfile   = os.path.join(RESULTS_DIR, file_name + ".jsonl")

model_device = next(model.parameters()).device

counts  = {g: 0 for g in ("ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT")}
em_hits = {g: 0 for g in ("ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT")}

import time
start_time = time.time()


def update_metrics(tier, em):
    counts["ALL"] += 1
    em_hits["ALL"] += em

    if tier in counts:
        counts[tier] += 1
        em_hits[tier] += em


def current_scores():
    return {
        "ALL_EM":     safe_div(em_hits["ALL"],        counts["ALL"]),
        "Long_Tail":  safe_div(em_hits["LONG-TAIL"],  counts["LONG-TAIL"]),
        "Infrequent": safe_div(em_hits["INFREQUENT"], counts["INFREQUENT"]),
        "Frequent":   safe_div(em_hits["FREQUENT"],   counts["FREQUENT"]),
    }


# -------------------------
# Helpers (SRE-aware context)
# -------------------------
def build_context_from_sre_list(sre_list, score_th: float, k: int, clip_chars: int = 250) -> str:
    if not isinstance(sre_list, list):
        return ""
    passed = []
    for d in sre_list:
        if not isinstance(d, dict):
            continue
        score = float(d.get("score", -1))
        txt   = (d.get("text") or "").strip()
        if txt and score >= score_th:
            passed.append(txt[:clip_chars])
        if len(passed) >= k:
            break
    return "\n\n".join(passed)

def get_context_for_run(ex: dict, run_type: str, use_context: bool) -> str:
    if not use_context:
        return ""

    rt = (run_type or "").upper()

    # SRE run: try retrieved_docs_sre with threshold, else fall back
    if "SRE" in rt:
        sre_list = ex.get(SRE_RETR_COL) or []
        ctx = build_context_from_sre_list(sre_list, score_th=SRE_SCORE_TH, k=TOP_K)
        if ctx.strip():
            return ctx

        base_docs = ex.get(BASE_RETR_COL) or []
        return build_context(base_docs, k=TOP_K)

    # non-SRE run: old behavior
    base_docs = ex.get(BASE_RETR_COL) or []
    return build_context(base_docs, k=TOP_K)

# -------------------------
# Run
# -------------------------
start_time = time.time()

# Clear any leftover bars (important if previous run crashed)
tqdm._instances.clear()

with (open(outfile, "w", encoding="utf-8", buffering=1) if WRITE_OUTPUTS else nullcontext()) as writer:

    with tqdm(
        total=len(ds),
        desc="Generating + Evaluating",
        dynamic_ncols=True,
        leave=False,   # removes bar after completion
        position=0
    ) as pbar:

        for i in range(len(ds)):
            ex = ds[i]

            q, s_pop = ex["question"], int(ex.get("s_pop", 0))
            tier = tier_from_spop(s_pop)

            gold = parse_list(ex.get("possible_answers"))
            gold_norm_set = {norm(g) for g in gold if norm(g)}

            # NEW: SRE-aware context selection
            context = get_context_for_run(ex, RUN_TYPE, USE_CONTEXT)

            pred = ask_llm_generate(
                model=model,
                tokenizer=tokenizer,
                question=q,
                context=context,
                use_context=USE_CONTEXT,
                device=model_device,
                print_prompt=PRINT_PROMPTS,
                prompt_fn=prompt_fn
            )

            pred_norm = norm(pred)
            em = int(pred_norm in gold_norm_set) if gold_norm_set else 0

            update_metrics(tier, em)

            if WRITE_OUTPUTS:
                record = { "i": i, "s_pop": s_pop, "tier": tier, "question": q,
                    "gold": gold, "pred": pred, "em": em
                }
                write_record(writer, record)

            pbar.update(1)

            if i % 10 == 0:
                pbar.set_postfix({k: f"{v:.4f}" for k, v in current_scores().items()})

total_time = time.time() - start_time

generate_report(
    counts,
    em_hits,
    file_name,
    model_name=MODEL_NAME,
    run_type=RUN_TYPE,
    total_time=total_time
)

Generating + Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

Saved report: results/openai_gpt-oss-20b-FUSEQA-SRE_20260226-2013.report.txt | Time=88.28s | ALL=0.3760 | LONG-TAIL=0.2308 | INFREQUENT=0.4318 | FREQUENT=0.2727


'results/openai_gpt-oss-20b-FUSEQA-SRE_20260226-2013.report.txt'

In [ ]:
! cat $(ls -t results/*.jsonl | head -1) | head -225

{"i": 0, "s_pop": 142, "tier": "INFREQUENT", "question": "What is George Rankin's occupation?", "gold": ["politician", "political leader", "political figure", "polit.", "pol"], "pred": "**Soldier**", "em": 0}
{"i": 1, "s_pop": 236, "tier": "INFREQUENT", "question": "What is John Mayne's occupation?", "gold": ["journalist", "journo", "journalists"], "pred": "Printer", "em": 0}
{"i": 2, "s_pop": 58, "tier": "LONG-TAIL", "question": "What is Henry Feilden's occupation?", "gold": ["politician", "political leader", "political figure", "polit.", "pol"], "pred": "Explorer", "em": 0}
{"i": 3, "s_pop": 127, "tier": "INFREQUENT", "question": "What is Kathy Saltzman's occupation?", "gold": ["politician", "political leader", "political figure", "polit.", "pol"], "pred": "We need to answer", "em": 0}
{"i": 4, "s_pop": 317, "tier": "INFREQUENT", "question": "What is Eleanor Davis's occupation?", "gold": ["cartoonist", "graphic artist", "animator", "illustrator"], "pred": "Cartoonist", "em": 1}
{"i":